<a href="https://colab.research.google.com/github/emilynchristiansen/NYC-Drinking-Fountains/blob/main/Repeatable_search_nyc_parks_amenities.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
import geopandas as gpd
from pathlib import Path
import requests
import pandas as pd

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving NYC_Parks_Drinking_Fountains_McCarren_Park_deduped.geojson to NYC_Parks_Drinking_Fountains_McCarren_Park_deduped.geojson


In [11]:
from google.colab import files
uploaded = files.upload()

Saving NYC_Parks_Drinking_Fountains_McCarren_Park_deduped.geojson to NYC_Parks_Drinking_Fountains_McCarren_Park_deduped.geojson


In [2]:
from google.colab import files
uploaded = files.upload()

Saving NYC_Parks_Drinking_Fountains_CanarsiePark.geojson to NYC_Parks_Drinking_Fountains_CanarsiePark.geojson


In [3]:
city_data = gpd.read_file("NYC_Parks_Drinking_Fountains_CanarsiePark.geojson")

In [12]:
city_data = gpd.read_file("NYC_Parks_Drinking_Fountains_McCarren_Park_deduped.geojson")

In [4]:
osm_query = '[out:json];area({area_id})->.searchArea;node["amenity"="drinking_water"](area.searchArea);out body;'

In [13]:
def query_overpass(query_string):
    """
    Send a query to the Overpass API with a proper User-Agent header.
    """
    url = "https://overpass-api.de/api/interpreter"
    headers = {"User-Agent": "NYC-Parks-Research/1.0 (student project)"}
    response = requests.post(url, data={"data": query_string}, headers=headers)
    response.raise_for_status()
    return response.json()


def compare_osm_to_city(
    area_id,
    city_data,
    osm_query,
    threshold_m=20,
    output_dir="output",
    label="analysis"
):
    # --- 3. Reproject city data ---
    city_data = city_data.to_crs(epsg=4326).copy()

    # --- 4. Fetch OSM data via Overpass ---
    area_info_query = f'[out:json];area({area_id});out tags;'
    area_info = query_overpass(area_info_query)
    park_name = area_info["elements"][0]["tags"].get("name", "Unknown")
    print(f"Querying park: {park_name} (area_id: {area_id})")

    formatted_query = osm_query.format(area_id=area_id)
    result = query_overpass(formatted_query)

    records = [
        {"osm_id": el["id"], "geometry": gpd.points_from_xy([el["lon"]], [el["lat"]])[0], **el.get("tags", {})}
        for el in result["elements"] if el["type"] == "node"
    ]

    if records:
        osm_data = gpd.GeoDataFrame(records, geometry="geometry", crs="EPSG:4326")
    else:
        osm_data = gpd.GeoDataFrame(columns=["osm_id", "geometry"], geometry="geometry", crs="EPSG:4326")

    print(f"OSM points fetched:    {len(osm_data)}")
    print(f"City points (clipped): {len(city_data)}")

    # --- 5. Reproject both to meters for distance calculation ---
    osm_proj  = osm_data.to_crs(epsg=3857)
    city_proj = city_data.to_crs(epsg=3857)

    # --- 6. Spatial join ---
    matched = gpd.sjoin_nearest(
        osm_proj, city_proj,
        how="inner",
        max_distance=threshold_m,
        distance_col="distance_m"
    )

    osm_only  = osm_proj[~osm_proj.index.isin(matched.index)].copy()
    city_only = city_proj[~city_proj.index.isin(matched["index_right"])].copy()

    # --- 7. Summary ---
    total_city = len(city_data)
    match_rate = round(len(matched) / total_city * 100, 1) if total_city > 0 else 0

    summary = {
        "osm_total":    len(osm_data),
        "city_total":   total_city,
        "matched":      len(matched),
        "osm_only":     len(osm_only),
        "city_only":    len(city_only),
        "match_rate_%": match_rate,
        "threshold_m":  threshold_m,
    }

    print("\n--- Summary ---")
    for k, v in summary.items():
        print(f"  {k}: {v}")

    # --- 8. Export ---
    Path(output_dir).mkdir(exist_ok=True)
    for name, gdf in [("matched", matched), ("osm_only", osm_only), ("city_only", city_only)]:
        path = Path(output_dir) / f"{label}_{name}.geojson"
        gdf.to_crs(epsg=4326).to_file(path, driver="GeoJSON")
        print(f"  Saved: {path}")

    # --- 9. Return ---
    return {
        "matched":  matched,
        "osm_only": osm_only,
        "city_only": city_only,
        "summary":  summary,
    }

In [15]:
results = compare_osm_to_city(
    area_id   = 3613902286,
    city_data = city_data,
    osm_query = osm_query,
    label     = "mccarren_park_test"
)

Querying park: McCarren Park (area_id: 3613902286)
OSM points fetched:    14
City points (clipped): 16

--- Summary ---
  osm_total: 14
  city_total: 16
  matched: 9
  osm_only: 5
  city_only: 7
  match_rate_%: 56.2
  threshold_m: 20
  Saved: output/mccarren_park_test_matched.geojson
  Saved: output/mccarren_park_test_osm_only.geojson
  Saved: output/mccarren_park_test_city_only.geojson


### Generating Table with Points

In [23]:
data = {
    "Park": ["Prospect Park", "McCarren Park", "Canarsie Park"],
    "NYC Parks Count": [75, 16, 14],
    "OSM Count": [24, 14, 0],
    "Matches": [21, 9, 0],
    "City-only": [54, 7, 14],
    "OSM-only": [3, 5, 0],
}

df = pd.DataFrame(data)
df["Match Rate (%)"] = (df["Matches"] / df["NYC Parks Count"] * 100).round(1)

df

,Park,NYC Parks Count,OSM Count,Matches,City-only,OSM-only,Match Rate (%)
0,Prospect Park,75,24,21,54,3,28.0
1,McCarren Park,16,14,9,7,5,56.2
2,Canarsie Park,14,0,0,14,0,0.0


In [20]:
# Export the DataFrame to a CSV file
df.to_csv('park_fountain_comparison_summary.csv', index=False)
print('DataFrame exported to park_fountain_comparison_summary.csv')

DataFrame exported to park_fountain_comparison_summary.csv


In [22]:
import matplotlib.pyplot as plt

# Create a figure and a set of subplots
fig, ax = plt.subplots(figsize=(10, 3)) # Adjust figure size as needed

# Hide axes
ax.xaxis.set_visible(False)
ax.yaxis.set_visible(False)
ax.set_frame_on(False)

# Create the table
table = ax.table(cellText=df.values,
                 colLabels=df.columns,
                 loc='center',
                 cellLoc='center')

# Adjust font size for better readability
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1.2, 1.2) # Adjust column width and row height

# Save the table as an image
plt.savefig('park_fountain_comparison_summary.png', bbox_inches='tight', dpi=300)
plt.close(fig) # Close the figure to prevent it from displaying in the notebook

print('DataFrame exported as park_fountain_comparison_summary.png')

DataFrame exported as park_fountain_comparison_summary.png
